In [1]:
# Import Required Libraries (PySpark-only, no pandas)
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, LongType, StringType, ArrayType, FloatType, IntegerType
from pyspark.sql.functions import explode, lower, col, collect_set, trim, regexp_replace, count, udf
import os

# In cluster use the HDFS path prefix
path_prefix = "hdfs:///projects/BDA-12/"
# In local use the local path prefix
# path_prefix = "../"

In [2]:
# Load ingredient and nutrient data
spark = SparkSession.builder.appName('HealthinessScoring').getOrCreate()

# Ingredient schema
ingredients_schema = StructType([
    StructField('fdc_id', LongType(), False),
    StructField('description', StringType(), True),
    StructField('all_ingredients', ArrayType(StringType()), True)
])
df_ing = spark.read.schema(ingredients_schema).parquet(
    f'{path_prefix}/output/ingredients_nutrional_profiles/'
)

# Nutrient columns (select a reasonable subset used by scoring)
nutri_cols = [
    'fdc_id', 'energy', 'protein', 'carbs', 'total_fat', 'fiber', 'sugars', 'sodium',
    'cholesterol', 'saturated_fat', 'vitamin_c', 'potassium', 'magnesium'
]
df_nutri = spark.read.parquet(f'{path_prefix}/output/nutritional_profiles')
df_nutri = df_nutri.select(*nutri_cols)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/15 17:50:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Explode and normalize ingredients, then join with nutrients using PySpark only
pattern = r'[\(\)\"\*\[\]\{\}\.,:;\'\-&]'
df_exploded = df_ing.select('fdc_id', 'description', explode('all_ingredients').alias('ingredient'))
df_exploded = df_exploded.withColumn('ingredient_norm', trim(lower(regexp_replace(col('ingredient'), pattern, ''))))
df_unique = df_exploded.groupBy('fdc_id', 'description').agg(collect_set('ingredient_norm').alias('ingredients_set'))

# Join with nutrient profiles (PySpark join)
foods = df_unique.join(df_nutri, on='fdc_id', how='left')

In [4]:
# Define combined scoring rules (pure Python functions used inside Spark UDFs)
healthy_ingredients = set([
    'whole wheat', 'oat', 'spinach', 'broccoli', 'carrot', 'olive oil', 'almond', 'quinoa', 'lentil', 'chicken breast', 'salmon', 'egg',
    'tomato', 'avocado', 'brown rice', 'beans', 'walnut', 'pumpkin seed', 'chia seed', 'flaxseed', 'greek yogurt', 'blueberry', 'apple', 'pear',
    'cabbage', 'cauliflower', 'zucchini', 'bell pepper', 'garlic', 'onion', 'sweet potato', 'kale', 'arugula', 'mushroom', 'cod', 'tuna', 'sardine',
    'hazelnut', 'cashew', 'pistachio', 'sunflower seed', 'sesame seed', 'turkey', 'beet', 'raspberry', 'strawberry', 'lemon', 'lime', 'orange'
])

unhealthy_ingredients = set([
    'sugar', 'corn syrup', 'palm oil', 'margarine', 'sodium benzoate', 'monosodium glutamate', 'artificial flavor', 'high fructose corn syrup', 'trans fat',
    'hydrogenated oil', 'shortening', 'dextrose', 'fructose', 'glucose syrup', 'aspartame', 'acesulfame k', 'saccharin', 'caramel color', 'red 40', 'yellow 5',
    'blue 1', 'potassium bromate', 'bht', 'bha', 'propyl gallate', 'propylene glycol', 'sorbitol', 'polysorbate 80', 'soy protein isolate', 'refined flour',
    'bleached flour', 'canola oil', 'vegetable oil', 'disodium inosinate', 'disodium guanylate', 'sodium nitrate', 'sodium nitrite', 'tbhq', 'phosphoric acid'
])

def score_ingredient(ingredient):
    if ingredient is None:
        return 0
    if ingredient in healthy_ingredients:
        return 1
    elif ingredient in unhealthy_ingredients:
        return -1
    else:
        return 0

def score_nutrients(energy, protein, fiber, sugars, sodium, total_fat, cholesterol, saturated_fat, vitamin_c, potassium, magnesium):
    score = 0.0
    if protein is not None:
        score += float(protein) * 0.5
    if fiber is not None:
        score += float(fiber) * 0.7
    if vitamin_c is not None:
        score += float(vitamin_c) * 0.05
    if potassium is not None:
        score += float(potassium) * 0.0005
    if magnesium is not None:
        score += float(magnesium) * 0.01
    if sugars is not None:
        score -= float(sugars) * 0.4
    if sodium is not None:
        score -= float(sodium) * 0.002
    if total_fat is not None:
        score -= float(total_fat) * 0.2
    if cholesterol is not None:
        score -= float(cholesterol) * 0.01
    if saturated_fat is not None:
        score -= float(saturated_fat) * 0.2
    return score

def nutri_score_func(energy, sugars, saturated_fat, sodium, fiber, protein, vitamin_c, potassium, magnesium):
    score = 0.0
    if energy is not None:
        score += float(energy) * 0.003
    if sugars is not None:
        score += float(sugars) * 0.5
    if saturated_fat is not None:
        score += float(saturated_fat) * 1.0
    if sodium is not None:
        score += float(sodium) * 0.004
    if fiber is not None:
        score -= float(fiber) * 1.2
    if protein is not None:
        score -= float(protein) * 0.8
    if vitamin_c is not None:
        score -= float(vitamin_c) * 0.05
    if potassium is not None:
        score -= float(potassium) * 0.001
    if magnesium is not None:
        score -= float(magnesium) * 0.02
    return score

def total_healthiness_score(ingredients_set, energy, protein, fiber, sugars, total_fat, sodium, cholesterol, saturated_fat, vitamin_c, potassium, magnesium):
    ing_score = 0.0
    if ingredients_set is not None:
        for ing in ingredients_set:
            ing_score += score_ingredient(ing)
    nutri_score_val = score_nutrients(energy, protein, fiber, sugars, sodium, total_fat, cholesterol, saturated_fat, vitamin_c, potassium, magnesium)
    nutri_score_external = nutri_score_func(energy, sugars, saturated_fat, sodium, fiber, protein, vitamin_c, potassium, magnesium)
    return float(ing_score + nutri_score_val - nutri_score_external)

In [5]:
# Filter foods with null or zero values for main 5 nutrients (PySpark)
main_fields = ['energy', 'protein', 'fiber', 'sugars', 'total_fat']
cond = None
for field in main_fields:
    this_cond = (col(field).isNotNull()) & (col(field) != 0)
    cond = this_cond if cond is None else (cond & this_cond)

if cond is not None:
    foods = foods.filter(cond)
else:
    foods = foods

# Register UDF and compute healthiness score column
total_udf = udf(total_healthiness_score, FloatType())
foods = foods.withColumn('healthiness_score', total_udf(
    col('ingredients_set'), col('energy'), col('protein'), col('fiber'), col('sugars'),
    col('total_fat'), col('sodium'), col('cholesterol'), col('saturated_fat'),
    col('vitamin_c'), col('potassium'), col('magnesium')
))

In [6]:
# Display top and bottom foods by combined score using PySpark actions
print('Top 40 Healthiest Foods (Combined Score):')
foods.orderBy(col('healthiness_score').desc()).select('description', 'healthiness_score', 'ingredients_set', 'protein', 'fiber', 'vitamin_c', 'potassium', 'magnesium', 'sugars', 'sodium', 'total_fat', 'cholesterol', 'saturated_fat').show(40, truncate=False)

print('Top 10 Least Healthy Foods (Combined Score):')
foods.orderBy(col('healthiness_score').asc()).select('description', 'healthiness_score', 'ingredients_set', 'protein', 'fiber', 'vitamin_c', 'potassium', 'magnesium', 'sugars', 'sodium', 'total_fat', 'cholesterol', 'saturated_fat').show(10, truncate=False)

Top 40 Healthiest Foods (Combined Score):


+---------------------------------------------------------------------------------+-----------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

+--------------------------------------------+-----------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+------+---------+---------+--

In [7]:
# Export scores
output_dir = f'{path_prefix}output/HealthinessScores'
# For local filesystem ensure directory exists; for HDFS the Spark writer will create the output
if 'hdfs' not in path_prefix:
    os.makedirs(output_dir, exist_ok=True)
# Write as CSV (will create a folder with part files).
foods.select('fdc_id', 'description', 'healthiness_score').coalesce(1).write.mode('overwrite').csv(output_dir)
print(f'Combined healthiness scores exported to {output_dir}')

Combined healthiness scores exported to hdfs:///projects/BDA-12/output/HealthinessScores
